In [ ]:
import torch
import model
import nerf_dataset_simple

Model=model.MLP(33,4,[64,64,128,64])
import torch.nn.functional as F

In [ ]:
data_dir = 'data/lego'

print(f"Solve the {data_dir} picture ...")
images, poses, hwf = nerf_dataset_simple.load_blender_data(data_dir, 'train')
H, W, focal = hwf
print(f"Loaded {len(images)} images, resolution: {H}x{W}, focal: {focal}")

def Simple_Light():
    rays_o, rays_d, target_rgb = nerf_dataset_simple.sample_random_pixels(images, poses, hwf)
    target_rgb=target_rgb[...,:3] # 忽略透明度
    mask = (target_rgb < 1e-4).all(dim=-1) 
    target_rgb[mask] = 1.0 # 背景变成白色
    return [rays_o, rays_d, target_rgb]

In [ ]:
import torch.nn as nn
import torch.optim as optim

loss_fn=nn.MSELoss()
optimizer = optim.Adam(Model.parameters(), lr=0.001,weight_decay=1e-4)
num_epochs = 500

for epoch in range(num_epochs):
    [rays_o,rays_d,target_rgb]=Simple_Light()
    logits=model.Calc_Light(rays_o,rays_d,Model)
    loss = loss_fn(logits, target_rgb)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [ ]:
import torch

torch.save(Model.state_dict(), 'nerf_test.pth')
print("模型已成功保存到本地！")